In [1]:
# import library
import pandas as pd 

In [2]:
#load data
df= pd.read_csv(r"C:\Users\tanma\Downloads\DATASETS\netflix_dataset\netflix_titles.csv")


In [3]:
# handling nulls

df['country'].fillna(df['country'].mode()[0], inplace=True)



C:\Users\tanma\AppData\Local\Temp\ipykernel_11616\485450747.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna(df['country'].mode()[0], inplace=True)


In [4]:
x = df.drop(['type', 'show_id', 'title', 'cast', 'director', 'description', 'date_added','listed_in','duration'], axis=1)
x = x.fillna(0) 
y = df['type']

In [5]:
x

,country,release_year,rating
0,United States,2020,PG-13
1,South Africa,2021,TV-MA
2,United States,2021,TV-MA
3,United States,2021,TV-MA
4,India,2021,TV-MA
...,...,...,...
8802,United States,2007,R
8803,United States,2018,TV-Y7
8804,United States,2009,R
8805,United States,2006,PG


In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer # Do different work on different columns in one step 
 # it will keep numerical column as it as but change the categorical columns to numerical by using one hot encoding 

cat_col = ["country","rating"]

# ❗ FIX 2: now duration is numeric so safe
num_col = ["release_year"]

preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_col)
    ],
    remainder='passthrough'
)

In [7]:
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline(steps = [
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000,class_weight='balanced'))
])


In [8]:
x['country'] = x['country'].astype(str)
x['rating'] = x['rating'].astype(str)

# ensure numeric
x['release_year'] = pd.to_numeric(x['release_year'], errors='coerce')
# x['duration'] = pd.to_numeric(x['duration'], errors='coerce')

# remove NaN
x = x.fillna(0)

In [9]:
x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=42
)

model.fit(x_train,y_train)

C:\Users\tanma\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'rating'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [10]:
from sklearn.metrics import accuracy_score,classification_report
y_pred = model.predict(x_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)
print("accuracy_score :", accuracy)
print("classification_report",classification_report(y_test, y_pred))


accuracy_score : 0.6838819523269013
classification_report               precision    recall  f1-score   support

       Movie       0.92      0.59      0.72      1214
     TV Show       0.50      0.89      0.64       548

    accuracy                           0.68      1762
   macro avg       0.71      0.74      0.68      1762
weighted avg       0.79      0.68      0.69      1762



In [73]:
from sklearn.tree import DecisionTreeClassifier
D_model = Pipeline(steps = [
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=8)) # here the classes are imbalanced because it is more good on movies but not in the tv shows 
])


In [74]:
D_model.fit(x_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'rating'])])),
                ('classifier', DecisionTreeClassifier(max_depth=8))])

In [75]:
Y_pred = D_model.predict(x_test)

In [76]:
accuracy = accuracy_score(y_test, Y_pred)
print("accuracy_score :", accuracy)
print("classification_report",classification_report(y_test, Y_pred))


accuracy_score : 0.7451759364358683
classification_report               precision    recall  f1-score   support

       Movie       0.77      0.89      0.83      1214
     TV Show       0.64      0.41      0.50       548

    accuracy                           0.75      1762
   macro avg       0.71      0.65      0.67      1762
weighted avg       0.73      0.75      0.73      1762



In [141]:
# random forest
from sklearn.ensemble import RandomForestClassifier
RF_model = Pipeline(steps = [
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300,max_depth=9,class_weight='balanced')) 
])

In [142]:
RF_model.fit(x_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'rating'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=9,
                                        n_estimators=300))])

In [143]:
Y_Pred = RF_model.predict(x_test)

In [144]:
accuracy = accuracy_score(y_test, Y_Pred)
print("accuracy_score :", accuracy)
print("classification_report",classification_report(y_test, Y_Pred))

accuracy_score : 0.7133938706015891
classification_report               precision    recall  f1-score   support

       Movie       0.91      0.65      0.76      1214
     TV Show       0.52      0.86      0.65       548

    accuracy                           0.71      1762
   macro avg       0.72      0.75      0.70      1762
weighted avg       0.79      0.71      0.72      1762

